In [ ]:
%%bash
set -e

echo "=== Step 1: Upgrade pip, setuptools, wheel ==="
pip install --upgrade pip setuptools wheel

echo "=== Step 2: Install PyTorch + CUDA (already preinstalled on Colab) ==="
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

echo "=== Step 3: Install Detectron2 from source ==="
pip install 'git+https://github.com/facebookresearch/detectron2.git'

echo "=== Step 4: Install helper dependencies for Mask2Former ==="
pip install git+https://github.com/facebookresearch/fvcore.git iopath==0.1.10 yacs pycocotools opencv-python tqdm requests

echo "=== Step 5: Verify PyTorch and Detectron2 ==="
python -c "
import torch
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Detectron2 import succeeded')"


In [ ]:
%%bash
set -e

echo "=== Step 1: Upgrade pip, setuptools, wheel ==="
pip install --upgrade pip setuptools wheel

echo "=== Step 2: Install PyTorch + CUDA (already preinstalled on Colab) ==="
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

echo "=== Step 3: Install Detectron2 from source ==="
pip install 'git+https://github.com/facebookresearch/detectron2.git'

echo "=== Step 4: Install helper dependencies for Mask2Former ==="
pip install git+https://github.com/facebookresearch/fvcore.git iopath==0.1.10 yacs pycocotools opencv-python tqdm requests

echo "=== Step 5: Verify PyTorch and Detectron2 ==="
python -c "
import torch
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
print('Torch version:', torch._version_)
print('CUDA available:', torch.cuda.is_available())
print('Detectron2 import succeeded')"

In [ ]:
import os
import requests
import cv2
import torch
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import MetadataCatalog
from google.colab.patches import cv2_imshow

# --------------------------
# 1) Download example image safely using requests
# --------------------------
image_path = "example.jpg"

im = cv2.imread(image_path)
if im is None:
    raise ValueError(f"Failed to read image from {image_path}")
im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

# --------------------------
# 2) Setup Detectron2 config
# --------------------------
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------
# 3) Create predictor
# --------------------------
predictor = DefaultPredictor(cfg)

# --------------------------
# 4) Run inference
# --------------------------
outputs = predictor(im)

# --------------------------
# 5) Visualize results
# --------------------------
v = Visualizer(im, metadata=MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2, instance_mode=ColorMode.IMAGE)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2_imshow(cv2.cvtColor(out.get_image(), cv2.COLOR_RGB2BGR))


In [ ]:
import torch
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

import cv2
import numpy as np
from google.colab.patches import cv2_imshow

from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import MetadataCatalog
import os

# --------------------------
# 1) Download example image
# --------------------------

image_path = "example.jpg"

# Load image
im = cv2.imread(image_path)
if im is None:
    raise ValueError(f"Failed to read image from {image_path}")
im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

# --------------------------
# 2) Setup Mask2Former config
# --------------------------
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask2former_R50_3x.yaml"
))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask2former_R50_3x.yaml"
)
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------
# 3) Create predictor
# --------------------------
predictor = DefaultPredictor(cfg)

# --------------------------
# 4) Run inference
# --------------------------
outputs = predictor(im)

# --------------------------
# 5) Visualize results
# --------------------------
metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])
v = Visualizer(im, metadata=metadata, scale=1.2, instance_mode=ColorMode.IMAGE)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

# Display image in Colab
cv2_imshow(cv2.cvtColor(out.get_image(), cv2.COLOR_RGB2BGR))


In [ ]:
# RUN THIS CELL in Colab (GPU runtime). Uses the Mask2Former you just built.
# It runs a producer-consumer pipeline: small prefetch -> single GPU consumer -> upload + checkpoint to Azure.
import os, time, json, logging, traceback, getpass, sys
from pathlib import Path
import threading
from queue import Queue, Empty
from typing import List, Dict, Any, Optional

import sys
sys.path.append("/content/Mask2Former")  # only needed if not installed
from mask2former import add_maskformer2_config

# ---------------- CONFIG ----------------
CONTAINER = "london"
PREFIX = "training/training/Images"
MANIFEST_BLOB = "manifests/mask2former_manifest.jsonl"
STATE_BLOB = "manifests/mask2former_state.json"
REMOTE_OUTPUT_PREFIX = "mask2former_results"
LOCAL_CACHE = Path("/tmp/m2f_cache")
LOCAL_OUT = Path("/tmp/m2f_out")
PREFETCH = 8           # keep small in Colab (8 is reasonable)
DOWNLOAD_WORKERS = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Pick the Mask2Former config and weights you prefer (Swin-base used as lighter example).
CFG_FILE = "Mask2Former/configs/coco/panoptic-segmentation/swin/maskformer2_swin_base_IN21k_384_bs16_100ep.yaml"
WEIGHTS = ("https://dl.fbaipublicfiles.com/maskformer/mask2former/coco/panoptic/"
           "maskformer2_swin_base_IN21k_384_bs16_100ep/model_final.pth")

# ---------------- logging ----------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("m2f_stream")

# ---------------- imports (Mask2Former + detectron2) ----------------
import torch, cv2, numpy as np
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import AzureError

from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import MetadataCatalog
# Mask2Former config helper already available from repo you built
from mask2former import add_maskformer2_config
from detectron2.projects.deeplab import add_deeplab_config

# ---------------- helpers ----------------
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def atomic_write_json(path: Path, obj: Dict[str,Any]):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, path)

# ---------------- Azure client (secure prompt) ----------------
AZ_CONN_STR = os.environ.get("AZ_CONN_STR")
if not AZ_CONN_STR:
    AZ_CONN_STR = getpass.getpass("Enter Azure Storage connection string (hidden): ").strip()
    # sanitize common paste errors
    if AZ_CONN_STR.startswith(("'", '"')) and AZ_CONN_STR.endswith(("'", '"')):
        AZ_CONN_STR = AZ_CONN_STR[1:-1].strip()
    AZ_CONN_STR = AZ_CONN_STR.replace("%22","")
    os.environ["AZ_CONN_STR"] = AZ_CONN_STR

blob_service = BlobServiceClient.from_connection_string(AZ_CONN_STR)
container_client = blob_service.get_container_client(CONTAINER)

def list_blobs_prefix(prefix: str) -> List[Dict[str, Any]]:
    out = []
    for b in container_client.list_blobs(name_starts_with=prefix):
        if b.name.rstrip("/") == prefix.rstrip("/"):
            continue
        out.append({"name": b.name, "size": getattr(b, "size", None)})
    return out

def upload_file_to_blob(blob_path: str, local_path: Path, overwrite=True):
    bc = container_client.get_blob_client(blob_path)
    with open(local_path, "rb") as fh:
        bc.upload_blob(fh, overwrite=overwrite)

def upload_bytes_to_blob(blob_path: str, data: bytes, overwrite=True):
    bc = container_client.get_blob_client(blob_path)
    bc.upload_blob(data, overwrite=overwrite)

def download_blob_to_local(blob_name: str, local_path: Path, expected_size: Optional[int]=None, retry=4):
    ensure_dir(local_path.parent)
    if local_path.exists() and expected_size is not None:
        try:
            if local_path.stat().st_size == expected_size:
                return "skipped"
        except Exception:
            pass
    part = local_path.with_suffix(local_path.suffix + ".part")
    attempt = 0
    while True:
        try:
            stream = container_client.download_blob(blob_name)
            with open(part, "wb") as fh:
                for chunk in stream.chunks():
                    fh.write(chunk)
            if expected_size is not None and part.stat().st_size != expected_size:
                raise RuntimeError(f"size mismatch after download {part}")
            os.replace(part, local_path)
            return "downloaded"
        except Exception as e:
            attempt += 1
            logger.warning("Download failed %s attempt=%d err=%s", blob_name, attempt, str(e))
            try:
                if part.exists() and attempt > 1:
                    part.unlink(missing_ok=True)
            except Exception:
                pass
            if attempt > retry:
                raise
            time.sleep(2 ** attempt)

# ---------------- build Mask2Former predictor ----------------
def build_predictor(cfg_file=CFG_FILE, weights=WEIGHTS, device="cuda"):
    cfg = get_cfg()
    add_deeplab_config(cfg)
    add_maskformer2_config(cfg)
    cfg.merge_from_file(cfg_file)
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.DEVICE = device if torch.cuda.is_available() else "cpu"
    cfg.MODEL.MASK_FORMER.TEST.SEMANTIC_ON = True
    cfg.MODEL.MASK_FORMER.TEST.INSTANCE_ON = True
    cfg.MODEL.MASK_FORMER.TEST.PANOPTIC_ON = True
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.0
    cfg.freeze()
    predictor = DefaultPredictor(cfg)
    return predictor, cfg

# ---------------- image processing (Mask2Former outputs) ----------------
def process_image_and_upload(predictor, cfg, blob_name: str, local_img_path: Path, remote_prefix: str) -> Dict[str,Any]:
    im = cv2.imread(str(local_img_path))
    if im is None:
        raise RuntimeError(f"Could not read {local_img_path}")
    outputs = predictor(im)

    saved = {}
    stem = local_img_path.stem
    ensure_dir(LOCAL_OUT)

    coco_meta = None
    try:
        if len(cfg.DATASETS.TEST) > 0:
            coco_meta = MetadataCatalog.get(cfg.DATASETS.TEST[0])
    except Exception:
        coco_meta = None

    # Panoptic visualization (if available)
    pan = outputs.get("panoptic_seg", None)
    if pan is not None:
        pan_seg, pan_info = pan
        viz = Visualizer(im[:, :, ::-1], coco_meta, scale=1.0, instance_mode=ColorMode.IMAGE_BW)
        pan_img = viz.draw_panoptic_seg(pan_seg.to("cpu"), pan_info).get_image()[:, :, ::-1]
        pan_path = LOCAL_OUT / f"{stem}_panoptic.png"
        cv2.imwrite(str(pan_path), pan_img)
        upload_file_to_blob(f"{remote_prefix}/panoptic/{stem}.png", pan_path, overwrite=True)
        saved["panoptic"] = f"{remote_prefix}/panoptic/{stem}.png"

    # Instance visualization, masks, metadata
    instances = outputs.get("instances", None)
    if instances is not None and len(instances) > 0:
        inst_cpu = instances.to("cpu")
        viz = Visualizer(im[:, :, ::-1], coco_meta, scale=1.0, instance_mode=ColorMode.IMAGE_BW)
        inst_img = viz.draw_instance_predictions(inst_cpu).get_image()[:, :, ::-1]
        inst_path = LOCAL_OUT / f"{stem}_instances.png"
        cv2.imwrite(str(inst_path), inst_img)
        upload_file_to_blob(f"{remote_prefix}/instances/{stem}.png", inst_path, overwrite=True)
        saved["instances_img"] = f"{remote_prefix}/instances/{stem}.png"

        pred_masks = inst_cpu.pred_masks.numpy() if hasattr(inst_cpu, "pred_masks") else None
        boxes_xyxy = inst_cpu.pred_boxes.tensor.numpy() if hasattr(inst_cpu, "pred_boxes") else None
        scores = inst_cpu.scores.numpy() if hasattr(inst_cpu, "scores") else None
        classes = inst_cpu.pred_classes.numpy() if hasattr(inst_cpu, "pred_classes") else None

        inst_meta = []
        for i in range(len(inst_cpu)):
            entry = {"id": int(i)}
            if boxes_xyxy is not None:
                entry["bbox_xyxy"] = boxes_xyxy[i].tolist()
            if scores is not None:
                entry["score"] = float(scores[i])
            if classes is not None:
                entry["class"] = int(classes[i])
            if pred_masks is not None:
                mask = (pred_masks[i] > 0).astype("uint8") * 255
                mask_local = LOCAL_OUT / f"{stem}_mask_{i}.png"
                cv2.imwrite(str(mask_local), mask)
                remote_mask_blob = f"{remote_prefix}/masks/{stem}_mask_{i}.png"
                upload_file_to_blob(remote_mask_blob, mask_local, overwrite=True)
                entry["mask"] = remote_mask_blob
            inst_meta.append(entry)
        inst_meta_blob = f"{remote_prefix}/metadata/{stem}_instances.json"
        upload_bytes_to_blob(inst_meta_blob, json.dumps(inst_meta).encode("utf8"), overwrite=True)
        saved["instances_meta"] = inst_meta_blob

    # Semantic visualization
    sem = outputs.get("sem_seg", None)
    if sem is not None:
        sem_cpu = sem.argmax(0).to("cpu").numpy()
        viz = Visualizer(im[:, :, ::-1], coco_meta, scale=1.0, instance_mode=ColorMode.IMAGE_BW)
        try:
            sem_img = viz.draw_sem_seg(torch.from_numpy(sem_cpu).to("cpu")).get_image()[:, :, ::-1]
            sem_path = LOCAL_OUT / f"{stem}_semantic.png"
            cv2.imwrite(str(sem_path), sem_img)
            upload_file_to_blob(f"{remote_prefix}/semantic/{stem}.png", sem_path, overwrite=True)
            saved["semantic"] = f"{remote_prefix}/semantic/{stem}.png"
        except Exception:
            logger.debug("semantic viz failed; skipping")

    # free memory
    try:
        del outputs, instances, sem, pan
    except Exception:
        pass
    torch.cuda.empty_cache()
    return saved

# ---------------- pipeline ----------------
def run_pipeline(prefetch=PREFETCH, download_workers=DOWNLOAD_WORKERS):
    ensure_dir(LOCAL_CACHE); ensure_dir(LOCAL_OUT)
    manifest = list_blobs_prefix(PREFIX)
    if not manifest:
        logger.error("No blobs under prefix %s", PREFIX); return
    # save/upload manifest
    lm = LOCAL_OUT / "manifest.jsonl"
    with open(lm, "w") as fh:
        for r in manifest: fh.write(json.dumps(r) + "\n")
    upload_file_to_blob(MANIFEST_BLOB, lm, overwrite=True)
    logger.info("Manifest uploaded (%d entries).", len(manifest))

    # load state
    try:
        bc = container_client.get_blob_client(STATE_BLOB)
        if bc.exists():
            state = json.loads(bc.download_blob().readall())
        else:
            state = {"processed": {}}
    except Exception:
        state = {"processed": {}}

    to_process = [r for r in manifest if not (r["name"] in state.get("processed", {}) and state["processed"][r["name"]].get("status")=="done")]
    logger.info("To process: %d", len(to_process))

    # queue
    q = Queue(maxsize=prefetch+4)

    def downloader_worker(items):
        for rec in items:
            blob_name = rec["name"]; expected_size = rec.get("size")
            if state.get("processed", {}).get(blob_name, {}).get("status") == "done":
                continue
            rel = Path(blob_name).relative_to(PREFIX)
            local_path = LOCAL_CACHE / rel
            ensure_dir(local_path.parent)
            try:
                res = download_blob_to_local(blob_name, local_path, expected_size)
                logger.info("Downloaded %s", blob_name)
            except Exception as e:
                logger.exception("Download failed: %s", str(e))
                state.setdefault("processed", {})[blob_name] = {"status":"download_failed", "error": str(e)}
                atomic_write_json(LOCAL_OUT / "state.json", state); upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
                continue
            while True:
                try:
                    q.put((blob_name, local_path), timeout=5); break
                except Exception:
                    time.sleep(0.5)
        logger.info("Downloader done.")

    # partition and start downloaders
    parts = [[] for _ in range(download_workers)]
    for i, rec in enumerate(to_process): parts[i % download_workers].append(rec)
    threads = []
    for i in range(download_workers):
        t = threading.Thread(target=downloader_worker, args=(parts[i],), daemon=True)
        t.start(); threads.append(t)

    # build predictor
    predictor, cfg = build_predictor(cfg_file=CFG_FILE, weights=WEIGHTS, device=("cuda" if torch.cuda.is_available() else "cpu"))
    logger.info("Predictor ready; processing images...")

    processed = 0; total = len(to_process)
    try:
        while True:
            try:
                blob_name, local_path = q.get(timeout=60)
            except Empty:
                all_done = all(not t.is_alive() for t in threads)
                if all_done and q.empty():
                    logger.info("All done"); break
                else:
                    continue
            if state.get("processed", {}).get(blob_name, {}).get("status") == "done":
                try: local_path.unlink(missing_ok=True)
                except Exception: pass
                q.task_done(); continue

            logger.info("Processing %d/%d: %s", processed+1, total, blob_name)
            try:
                saved = process_image_and_upload(predictor, cfg, blob_name, local_path, REMOTE_OUTPUT_PREFIX)
                state.setdefault("processed", {})[blob_name] = {"status":"done", "ts": time.time(), "saved": saved}
                atomic_write_json(LOCAL_OUT / "state.json", state)
                upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
                upload_bytes_to_blob(f"{REMOTE_OUTPUT_PREFIX}/done/{Path(local_path).stem}.json", json.dumps({"blob": blob_name, "ts": time.time()}).encode("utf8"), overwrite=True)
                logger.info("Uploaded outputs for %s", blob_name)
            except Exception as e:
                logger.exception("Processing failed for %s: %s", blob_name, str(e))
                state.setdefault("processed", {})[blob_name] = {"status":"processing_failed", "error": str(e)}
                atomic_write_json(LOCAL_OUT / "state.json", state)
                upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
            finally:
                try: local_path.unlink(missing_ok=True)
                except Exception: pass
                processed += 1
                q.task_done()
    except KeyboardInterrupt:
        logger.warning("Interrupted; saving state.")
    finally:
        for t in threads: t.join(timeout=1)
        atomic_write_json(LOCAL_OUT / "state.json", state)
        upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
        logger.info("Pipeline finished. Processed=%d", processed)

# ---------------- run ----------------
if __name__ == "__main__":
    run_pipeline(prefetch=PREFETCH, download_workers=DOWNLOAD_WORKERS)


In [ ]:
import torch
print(torch.__version__, torch.version.cuda)


In [ ]:
%%bash
cd /content/Mask2Former/mask2former/modeling/pixel_decoder/ops

# Build the custom CUDA extension
sh make.sh
